In [6]:
from pystac import Item
from odc.stac import load
from ldn.utils import PREDICTION_VERSION

# year = 2020
# year = 2010
# year = 2025
# year = 2000

# tile = "063/020" # Fiji
tile = "058/043"  # Kiribati
# tile = "185/125" # Cape Verde

# year = 2013
# tile = "066/022" # Fiji 2
# tile = "119/126"  # Belize

# year = 2011
# tile = "251/088" # Comoros
# tile = "312/105" # Singapore 1
# tile = "312/106" # Singapore 2

# year = 2001
# tile = "152/110"  # Suriname

# Rarotonga
# tile = "089/016"  # Rarotonga
# year = 2020
# year = 2024
year = 2023

# url = f"https://s3.us-west-2.amazonaws.com/data.ldn.auspatious.com/ausp_ls_geomad/{GEOMAD_VERSION}/{tile}/{year}/ausp_ls_geomad_{tile.replace('/', '_')}_{year}.stac-item.json"
url = f"https://s3.us-west-2.amazonaws.com/data.ldn.auspatious.com/ausp_ls_lulc_prediction/{PREDICTION_VERSION}/{tile.replace('/', '_')}/{year}/ausp_ls_lulc_prediction_{tile.replace('/', '_')}_{year}.stac-item.json"
print(url)
item = Item.from_file(url)
data = load([item])

https://s3.us-west-2.amazonaws.com/data.ldn.auspatious.com/ausp_ls_lulc_prediction/0-0-2/058_043/2023/ausp_ls_lulc_prediction_058_043_2023.stac-item.json


In [14]:
# Getting around S3 cached files being stale.
# import rioxarray
# import xarray as xr

# bands = {}
# for band in ["red", "green", "blue"]:
#     bands[band] = rioxarray.open_rasterio(
#         f"/Users/wj/Downloads/ausp_ls_geomad_063_020_2020_{band}.tif"
#     ).squeeze("band", drop=True)

# rgb = xr.concat(
#     [bands["red"], bands["green"], bands["blue"]],
#     dim="band"
# ).assign_coords(band=["red", "green", "blue"])

# # correct selection
# rgb = rgb.sel(band=["red", "green", "blue"])

# # per-band stats
# for band in rgb.band.values:
#     da = rgb.sel(band=band)
#     valid = da.where(da != 0)
#     lo = float(valid.quantile(0.001).values)
#     hi = float(valid.quantile(0.999).values)
#     print(f"{band}: 0.1%={lo:.0f}, 99.9%={hi:.0f}, range={hi - lo:.0f}")

# # global stretch
# all_valid = rgb.where(rgb != 0)
# vmin = float(all_valid.quantile(0.001).values)
# vmax = float(all_valid.quantile(0.999).values)

# print(f"\nGlobal stretch: vmin={vmin:.0f}, vmax={vmax:.0f}")

# data = rgb
# rgb = rgb.where(rgb != 0)

# img = rgb.to_dataset(dim="band")
# img.odc.explore(vmin=vmin, vmax=16000)

In [ ]:
# data[["red", "green", "blue"]].squeeze().to_array().plot.imshow(vmin=7500, vmax=12500, size=10)
# data.odc.explore()

In [ ]:
# from odc.geo.xr import write_cog

# rgb = data[["red", "green", "blue"]].to_array(dim="band")
# # Select the first time slice if time exists
# if "time" in rgb.dims:
#     rgb = rgb.isel(time=0)
# write_cog(rgb, "/tmp/rgb_cog_fiji_am_2013.tif", overwrite=True)


rgb = data[["red", "green", "blue"]]
for band in rgb.data_vars:
    valid = rgb[band].where((rgb[band] != 0))
    lo = float(valid.quantile(0.001).values)
    hi = float(valid.quantile(0.999).values)
    print(f"{band}: 0.1%={lo:.0f}, 99.9%={hi:.0f}, range={hi - lo:.0f}")

all_valid = rgb.to_array().where((rgb.to_array() != 0))
vmin = float(all_valid.quantile(0.001).values)
vmax = float(all_valid.quantile(0.999).values)
print(f"\nGlobal stretch: vmin={vmin:.0f}, vmax={vmax:.0f}")
# data.odc.explore(vmin=vmin, vmax=vmax)
data.odc.explore(vmin=7800, vmax=13000)
# data.odc.explore(vmin=vmin, vmax=16000)
# data.odc.explore(robust=True)

In [ ]:
count = data["count"]
print(count.min().values, count.max().values)
count.odc.explore(bands=["count", "count", "count"], cmap="gray")

In [ ]:
# data[["red", "green", "blue"]].squeeze().to_array().odc.write_cog("temp.tif")

In [10]:
data

<xarray.Dataset> Size: 31MB
Dimensions:                     (y: 3200, x: 3200, time: 1)
Coordinates:
  * y                           (y) float64 26kB 2.24e+05 2.24e+05 ... 1.28e+05
  * x                           (x) float64 26kB 2.568e+06 ... 2.664e+06
  * time                        (time) datetime64[ns] 8B 2023-01-01
    spatial_ref                 int32 4B 3832
Data variables:
    classification              (time, y, x) uint8 10MB 6 6 6 6 6 ... 6 6 6 6 6
    classification_unfiltered   (time, y, x) uint8 10MB 6 6 6 6 6 ... 6 6 6 6 6
    classification_probability  (time, y, x) uint8 10MB 99 99 99 99 ... 89 86 69

In [ ]:
# For prediction
from ldn.typology import colors
import numpy as np
from matplotlib.colors import ListedColormap

# present_classes = sorted(np.unique(data["classification"].values).tolist())

# print("Present classes:", present_classes)

# cmap = ListedColormap([colors[c] for c in present_classes])

# print(cmap.colors)

# data["classification"].odc.explore(
#     column="classification",
#     categorical=True,
#     categories=present_classes,
#     cmap=cmap,
# )


present_classes = sorted(np.unique(data["classification"].values).tolist())
display_classes = [c for c in present_classes if c != 255]

# Remap to 0-based sequential values
remapped = data["classification"].copy()
for i, c in enumerate(display_classes):
    remapped = remapped.where(data["classification"] != c, other=i)

cmap = ListedColormap([colors[c] for c in display_classes])

remapped.odc.explore(
    cmap=cmap,
    vmin=0,
    vmax=len(display_classes) - 1,
)